# PCA-based imputation

`pca_imputation` fills missing numeric values by repeatedly fitting PCA to a low-rank reconstruction. It is most useful when columns share a strong linear structure; it is not a replacement for investigating why process data are missing.

## 1. Create a reproducible low-rank data set

The three measurements below come from two latent process factors plus small measurement noise. We retain the complete data only to assess the imputed entries; in a real workflow those values would be unavailable.

In [1]:
import numpy as np
import pandas as pd

from pca_tools import pca_imputation

rng = np.random.default_rng(42)
n_samples = 120
latent = rng.normal(size=(n_samples, 2))
loadings = np.array([[1.0, 0.4], [0.8, -0.3], [-0.5, 1.2]])
complete_data = pd.DataFrame(
    latent @ loadings.T + rng.normal(scale=0.05, size=(n_samples, 3)),
    columns=["temperature", "pressure", "flow"],
)
complete_data.head()

,temperature,pressure,flow
0,-0.155120,0.551056,-1.488226
1,1.053325,0.424654,0.689081
2,-2.526746,-1.078329,-0.441844
3,-0.057235,0.178733,-0.426334
4,-0.271584,0.193129,-1.027516


## 2. Hide some measurements

Each column still needs at least one observed value. The function preserves observed values exactly and estimates only the missing cells.

In [2]:
data_with_gaps = complete_data.copy()
missing_mask = rng.random(data_with_gaps.shape) < 0.15
data_with_gaps = data_with_gaps.mask(missing_mask)

data_with_gaps.isna().sum().rename("missing values")

temperature    20
pressure       20
flow           19
Name: missing values, dtype: int64

## 3. Impute with two components

Set `n_components` to the number of latent directions supported by process knowledge or validation. Here it is two because the example was generated from two factors. The function first mean-fills gaps, standardizes columns, and iterates PCA reconstruction until convergence (or `max_iter`).

In [3]:
imputed_data = pca_imputation(
    data_with_gaps,
    n_components=2,
    max_iter=100,
    tol=1e-6,
)

imputed_data.head()

,temperature,pressure,flow
0,-0.155120,0.551056,-1.437673
1,1.053325,0.424654,0.689081
2,-2.526746,-1.283502,-0.441844
3,-0.175394,0.087282,-0.426334
4,-0.271584,0.193129,-1.027516


## 4. Check the result

For this controlled example we can compare only the deliberately hidden entries with their known values. The assertions are useful checks in production too: imputation should remove nulls and must not alter observed data.

In [4]:
assert not imputed_data.isna().any().any()
assert imputed_data.where(~missing_mask).equals(data_with_gaps.where(~missing_mask))

rmse = np.sqrt(
    np.mean((imputed_data.to_numpy()[missing_mask] - complete_data.to_numpy()[missing_mask]) ** 2)
)
print(f"RMSE on hidden values: {rmse:.3f}")

comparison = pd.DataFrame({
    "original": complete_data.to_numpy()[missing_mask],
    "imputed": imputed_data.to_numpy()[missing_mask],
})
comparison.head(10)

RMSE on hidden values: 0.291


,original,imputed
0,-1.488226,-1.437673
1,-1.078329,-1.283502
2,-0.057235,-0.175394
3,0.178733,0.087282
4,0.474844,0.510171
5,0.110473,0.113070
6,0.008775,0.069448
7,-0.148839,-0.142463
8,1.955505,1.968954
9,0.918444,0.899386


## Takeaway

PCA imputation borrows information from correlated variables, so a low-rank choice can improve on independent column-mean filling. Fit any downstream PCA monitoring model only after imputing its input, and use the same documented missing-data policy consistently for Phase I and Phase II data.